# SNR and Intelligibility

This notebook extracts the centerpiece of the legacy audio demo: the point where AM and FM experience the same noise and sound very different. It also keeps the impulse-noise comparison because intelligibility is about what your ears can still decode, not just a plotted SNR.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


In [ ]:
VOICE_FILE = ROOT / "assets" / "local" / "my_voice.m4a"
WORK_FS = 96_000
PLAY_FS = 44_100

if VOICE_FILE.exists():
    raw_fs, raw_audio = load_audio(VOICE_FILE, normalize_audio=True)
    raw_audio = normalize(ensure_mono(raw_audio))
    raw_audio = raw_audio[: int(raw_fs * 5)]
    voice_work = normalize(resample_signal(raw_audio, raw_fs, WORK_FS))
    voice_play = normalize(resample_signal(raw_audio, raw_fs, PLAY_FS))
    print(f"Loaded {VOICE_FILE.name} at {raw_fs} Hz")
else:
    t_fallback = np.arange(0, 3.0, 1 / WORK_FS)
    voice_work = normalize(
        0.7 * np.sin(2 * np.pi * 220 * t_fallback)
        + 0.4 * np.sin(2 * np.pi * 440 * t_fallback)
        + 0.2 * np.sin(2 * np.pi * 880 * t_fallback)
    )
    voice_play = normalize(resample_signal(voice_work, WORK_FS, PLAY_FS))
    print(f"No local voice recording found at {VOICE_FILE}. Using a synthetic fallback.")

t_work = np.arange(len(voice_work)) / WORK_FS


## Same Noise, Different Modulation

This is the main A/B comparison from the old notebook. We band-limit the message, modulate it both ways, inject the same noise realization, then compare the recovered audio.

In [ ]:
message = signal.sosfilt(signal.butter(5, [300, 3000], btype="band", fs=WORK_FS, output="sos"), voice_work)
message = normalize(message)
carrier_freq = 20_000
am_signal = am_modulate(message, carrier_freq=carrier_freq, fs=WORK_FS, mod_index=0.8)
fm_signal = fm_modulate(message, carrier_freq=carrier_freq, fs=WORK_FS, freq_dev=2_500)


In [ ]:
audio_out = audio_output_widget()
fig, axes = plt.subplots(2, 2, figsize=(13, 7))

def update_snr(snr_db=20.0):
    am_noisy, _ = add_awgn(am_signal, snr_db=snr_db, seed=42)
    fm_noisy, _ = add_awgn(fm_signal, snr_db=snr_db, seed=42)
    am_demod = am_demodulate(am_noisy, fs=WORK_FS)
    fm_demod = fm_demodulate(fm_noisy, fs=WORK_FS)

    for ax in axes.flat:
        ax.clear()

    plot_waveform(am_noisy[:6000], fs=WORK_FS, ax=axes[0, 0], title="AM + Noise")
    plot_waveform(fm_noisy[:6000], fs=WORK_FS, ax=axes[0, 1], title="FM + Noise", color="tab:orange")
    plot_spectrum(am_demod, fs=WORK_FS, ax=axes[1, 0], title="AM Demodulated Spectrum")
    plot_spectrum(fm_demod, fs=WORK_FS, ax=axes[1, 1], title="FM Demodulated Spectrum", color="tab:orange")
    for ax in axes[1]:
        ax.set_xlim(0, 6000)
        ax.set_ylim(-100, 5)
    fig.canvas.draw_idle()

    with audio_out:
        audio_out.clear_output(wait=True)
        display(Markdown(f"**AM demodulated, SNR={snr_db:.0f} dB**"))
        display(audio_player(resample_signal(am_demod, WORK_FS, PLAY_FS), rate=PLAY_FS))
        display(Markdown(f"**FM demodulated, SNR={snr_db:.0f} dB**"))
        display(audio_player(resample_signal(fm_demod, WORK_FS, PLAY_FS), rate=PLAY_FS))

controls = widgets.interactive(
    update_snr,
    snr_db=float_slider(min_value=0, max_value=30, step=1, value=20, description="SNR dB"),
)
display(controls, audio_out)


## Impulse Noise

Impulse noise is a realistic stress test for land-mobile audio. AM treats those spikes as part of the amplitude envelope; FM largely rejects them unless they are strong enough to disrupt phase tracking.

In [ ]:
am_impulse, _ = add_impulse_noise(am_signal, fs=WORK_FS, rate=80, amplitude=4.0, seed=7)
fm_impulse, _ = add_impulse_noise(fm_signal, fs=WORK_FS, rate=80, amplitude=4.0, seed=7)
am_impulse_demod = am_demodulate(am_impulse, fs=WORK_FS)
fm_impulse_demod = fm_demodulate(fm_impulse, fs=WORK_FS)

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
plot_waveform(am_impulse[:10_000], fs=WORK_FS, ax=axes[0], title="AM + Impulse Noise")
plot_waveform(fm_impulse[:10_000], fs=WORK_FS, ax=axes[1], title="FM + Impulse Noise", color="tab:orange")
plt.tight_layout()

display(Markdown("**AM with impulse noise**"))
display(audio_player(resample_signal(am_impulse_demod, WORK_FS, PLAY_FS), rate=PLAY_FS))
display(Markdown("**FM with impulse noise**"))
display(audio_player(resample_signal(fm_impulse_demod, WORK_FS, PLAY_FS), rate=PLAY_FS))


## What to Try

- Step SNR from 30 dB down to 0 dB and listen for when AM becomes tiring before it becomes fully unintelligible.
- Compare the spectral tilt of the demodulated FM noise to the flatter AM case.
- Listen to the impulse-noise examples and decide which artifacts are easier for your ear to ignore.

## Key Takeaway

Intelligibility is the human consequence of SNR. The reason FM sounds more robust is not marketing language; it is a direct result of how the modulation stores information.